In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm

from combra import data, angles

# Grid data generation

In [ ]:
types_dict = {
    "Ultra_Co11": "средние зерна",
    "Ultra_Co25": "мелкие зерна",
    "Ultra_Co8": "средне-мелкие зерна",
    "Ultra_Co6_2": "крупные зерна",
    "Ultra_Co15": "средне-мелкие зерна",
}

# max_img_per_class_list = [500, 5_000, 10_000]
max_img_per_class_list = [ 5_000]
min_segment_len = 5.0
output_dir = Path("./grid_results")
output_dir.mkdir(exist_ok=True)

# sources = [
#     # diff
#     '/home/david/mnt/ssd_2_sata/python/phd/wc_cv/co_angles/data/combined/gen_diff_768x768_N5000.h5',
#     '/home/david/mnt/ssd_2_sata/python/phd/wc_cv/co_angles/data/separeted/gen_diff_512x512_N500',
#     '/home/david/mnt/ssd_2_sata/python/phd/wc_cv/co_angles/data/separeted/gen_diff_256x256_N500',
#     # san
#     '/home/david/mnt/ssd_2_sata/python/phd/wc_cv/co_angles/data/combined/gen_san_512x512_N100_000.h5',
#     '/home/david/mnt/ssd_2_sata/python/phd/wc_cv/co_angles/data/combined/gen_san_256x256_N100_000.h5',
#     # orig
#     '/home/david/mnt/ssd_2_sata/python/phd/wc_cv/co_angles/data/separeted/orig_bc_left',
# ]

sources = [
    # diff
    '/home/david/mnt/ssd_2_sata/python/phd/wc_cv/co_angles/data/combined/gen_diff_768x768_N5000.h5',

]

for max_img_per_class in tqdm(max_img_per_class_list):
    for source_path in tqdm(sources):
        source_name = Path(source_path).stem
        save_path = output_dir / f"{source_name}_msl{int(min_segment_len)}"

        dataset = data.PobeditDataset(path=source_path, max_images_num_per_class=max_img_per_class)

        out = dataset.generate_angles(
            save_path=str(save_path),
            types_dict=types_dict,
            step=[0.1, 0.5, 1, 2, 3, 4, 5],
            workers=20,
            angles_tol=3,
            min_segment_len=min_segment_len,
        )

        print(f"[{source_name}, N={max_img_per_class}] -> {out}")

# Grid of plots

In [ ]:
# Grid plot: orig vs GAN vs diff at each (resolution × grain class).
# `combra.angles.angles_plot_grid` does the rendering; this cell only builds
# the per-cell trace spec.

GRID_DIR = Path('./grid_results')

# GAN parquets use class_0/1/2 instead of Ultra_Co* names.
SAN_NAME_FOR = {
    'Ultra_Co25': 'class_0',
    'Ultra_Co11': 'class_1',
    'Ultra_Co6_2': 'class_2',
}

# (source_type, resolution) -> sub-folder name. None resolution = single folder.
FOLDER_MAP = {
    ('diff', 256): 'gen_diff_256x256_N500_msl5',
    ('diff', 512): 'gen_diff_512x512_N500_msl5',
    ('diff', 768): 'gen_diff_768x768_N5000_msl5',
    ('gan',  256): 'gen_san_256x256_N100_000_msl5',
    ('gan',  512): 'gen_san_512x512_N100_000_msl5',
    ('orig', None): 'orig_bc_left_msl5',
}

STYLES = {
    'orig': dict(color='blue',   marker='circle'),
    'gan':  dict(color='orange', marker='square'),
    'diff': dict(color='green',  marker='triangle-down'),
}

resolutions = [256, 512, 768]
grain_classes = [
    ('Ultra_Co25', 'мелкие зерна'),
    ('Ultra_Co11', 'средние зерна'),
    ('Ultra_Co6_2', 'крупные зерна'),
]

step = 2
save = True

for max_n in [500, 5_000, 10_000]:
    grid = []
    for res in resolutions:
        row = []
        for class_key, _ in grain_classes:
            cell = []

            # Real (orig) — always angles_n90, fixed source.
            cell.append({
                'parquet': str(GRID_DIR / FOLDER_MAP[('orig', None)] / 'angles_n90.parquet'),
                'class_name': f'class_{class_key}',
                'label': 'orig', **STYLES['orig'],
            })

            # GAN — only available for 256/512.
            if res != 768:
                cell.append({
                    'parquet': str(GRID_DIR / FOLDER_MAP[('gan', res)] / f'angles_n{max_n}.parquet'),
                    'class_name': SAN_NAME_FOR[class_key],
                    'label': 'gan', **STYLES['gan'],
                })

            # Diff.
            cell.append({
                'parquet': str(GRID_DIR / FOLDER_MAP[('diff', res)] / f'angles_n{max_n}.parquet'),
                'class_name': f'class_{class_key}',
                'label': 'diff', **STYLES['diff'],
            })

            row.append(cell)
        grid.append(row)

    angles.angles_plot_grid(
        grid=grid,
        row_titles=[f'{r}×{r}' for r in resolutions],
        col_titles=[label for _, label in grain_classes],
        step=step,
        title=f'Распределения углов (step={step}, N изобр. на класс={max_n})',
        save=f'angles_grid_n{max_n}_step{step}.png' if save else None,
    )